In [1]:
#!/usr/bin/env python3

import requests
from bs4 import BeautifulSoup
import time
import random
import json
from urllib.parse import urljoin


# ------------------ GLOBAL SETTINGS ------------------

MAX_ARTICLES = 1000
OUT_FILE = "news_test_malayalam.jsonl"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (AcademicResearchBot/1.0)",
    "Accept-Language": "te-IN,ta-IN,hi-IN,bn-IN,en;q=0.8"
}

MIN_DELAY = 1.0
MAX_DELAY = 2.0


# ------------------ NEWSPAPER CONFIG ------------------

NEWSPAPERS = {

    # ---------------- KANNADA ----------------
"kannada_wikipedia": {
    "source": "wikipedia-kn",
    "base_url": "https://kn.wikipedia.org/wiki/ವರ್ಗ:ಭಾರತ",
    "article_pattern": "/wiki/",
    "language": "kn",
    "section": "india",
    "stop_markers": [
        "ಉಲ್ಲೇಖಗಳು",
        "ಬಾಹ್ಯ ಕೊಂಡಿಗಳು",
        "ಇನ್ನೂ ನೋಡಿ",
        "ವಿಕಿಪೀಡಿಯ"
    ]
},


    # ---------------- ODIA ----------------
    "odia_sambad": {
        "source": "sambad",
        "base_url": "https://sambad.in/india-and-beyond",
        "article_pattern": "/india-and-beyond/",
        "language": "or",
        "section": "india",
        "stop_markers": [
            "ଆହୁରି ପଢ଼ନ୍ତୁ",
            "Advertisement",
            "©",
            "Sambad.in"
        ]
    },

    # ---------------- MALAYALAM ----------------
 "malayalam_deshabhimani": {
    "source": "deshabhimani-ml",
    "base_url": "https://www.deshabhimani.com/category/national/",
    "article_pattern": "/national/",
    "language": "ml",
    "section": "india",
    "stop_markers": [
        "കൂടുതൽ വായിക്കുക",
        "Advertisement",
        "©",
        "deshabhimani.com",
        "ബന്ധപ്പെട്ട വാർത്ത",
        "വിജ്ഞാപനം"
    ]
},
    "keralakaumudi_politics": {
        "source": "keralakaumudi",
        "base_url": "https://keralakaumudi.com/news/mobile/sub-section.php?cid=9&sid=76",
        "article_pattern": "news.php?id=",
        "language": "ml",
        "section": "politics",
        "stop_markers": ["KAUMUDI", "©", "Read More"]
    },
"doolnews_politics": {
        "source": "doolnews",
        "base_url": "https://www.doolnews.com/tag/kerala-politics",
        "article_pattern": "doolnews.com",
        "language": "ml",
        "section": "politics",
        "stop_markers": ["©", "DoolNews"]
    },
    # ---------------- PUNJABI ----------------
    "punjabi_punjabkesari_epaper": {
    "source": "punjab-kesari-epaper",
    "base_url": "https://epaper.punjabkesari.in/",
    "article_pattern": ".html",
    "language": "pa",
    "section": "punjab-politics",
    "stop_markers": [
        "Advertisement",
        "©",
        "Punjab Kesari"
    ]
},


    
# ---------------- ASSAMESE (POLITICS) ----------------
"assamese_pratidintime": {
        "source": "pratidin-time",
        "base_url": "https://pratidintime.com/category/assam",
        "article_pattern": "/assam/",
        "language": "as",
        "section": "assam-politics",
        "stop_markers": ["আৰু পঢ়ক", "Advertisement", "©", "pratidintime.com"]
    },
"niyomiya_barta_assam": {
        "source": "niyomiya-barta",
        "base_url": "https://niyomiyabarta.com/category/assam/",
        "article_pattern": "niyomiyabarta.com/",
        "language": "as",
        "section": "politics",
        "stop_markers": ["Related Articles", "Share", "©", "ডিজিটেল ডেস্ক"]
    },
    
    
    # ---------------- ENGLISH ----------------
    "english_thehindu": {
        "source": "the-hindu",
        "base_url": "https://www.thehindu.com/news/national/",
        "article_pattern": ".ece",
        "language": "en",
        "section": "national",
        "stop_markers": [
            "READ MORE",
            "Advertisement",
            "©",
            "thehindu.com"
        ]
    }
}


# ------------------ HELPER FUNCTIONS ------------------

def fetch(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code == 200:
            return r.text
    except Exception:
        return None
    return None


def gather_article_links(html, paper):
    soup = BeautifulSoup(html, "html.parser")
    links = set()

    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if paper["article_pattern"] in href:
            links.add(urljoin(paper["base_url"], href))

    return list(links)


def extract_body(soup, stop_markers):
    texts = []

    for p in soup.find_all("p"):
        txt = p.get_text(" ", strip=True)
        if not txt:
            continue

        # HARD STOP on footer
        if any(marker in txt for marker in stop_markers):
            break

        if len(txt) > 30:
            texts.append(txt)

    return "\n\n".join(texts).strip()


def extract_title(soup):
    og = soup.find("meta", property="og:title")
    if og and og.get("content"):
        return og["content"].strip()

    if soup.title:
        return soup.title.get_text(strip=True)

    return ""


def scrape_article(url, paper):
    html = fetch(url)
    if not html:
        return None

    soup = BeautifulSoup(html, "html.parser")

    body = extract_body(soup, paper["stop_markers"])
    if not body:
        return None

    title = extract_title(soup)

    return {
        "url": url,
        "title": title,
        "date": "",
        "author": "",
        "body": body,
        "tags": [],
        "source": paper["source"],
        "language": paper["language"],
        "section": paper["section"]
    }


def save_jsonl(filename, record):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# ------------------ MAIN ------------------

def main():
    paper = NEWSPAPERS["keralakaumudi_politics"]

    print(f"[INFO] Fetching homepage: {paper['base_url']}")
    html = fetch(paper["base_url"])
    if not html:
        print("[ERROR] Failed to fetch homepage")
        return

    links = gather_article_links(html, paper)
    print(f"[INFO] Found {len(links)} article links")

    collected = 0

    for url in links:
        if collected >= MAX_ARTICLES:
            print("[INFO] Reached max article limit.")
            break

        time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

        try:
            article = scrape_article(url, paper)
            if article:
                save_jsonl(OUT_FILE, article)
                collected += 1
                print(f"[SAVED] {article['title'][:60]}")
            else:
                print(f"[SKIP] {url}")

        except Exception as e:
            print(f"[ERROR] {url} -> {e}")

    print(f"\nDone. Collected {collected} articles.")
    print(f"Output file: {OUT_FILE}")


if __name__ == "__main__":
    main()

[INFO] Fetching homepage: https://keralakaumudi.com/news/mobile/sub-section.php?cid=9&sid=76
[INFO] Found 0 article links

Done. Collected 0 articles.
Output file: news_test_malayalam.jsonl
